# Mamba: State Space Model — 실습 코드 2: SSM (State Space Model) 직접 구현
### 📘 상세 설명판 (처음 배우는 분들을 위한 주석 강화 버전)

- Tutorial ID: `expand-mamba-ssm`
- Tutorial: Mamba: State Space Model
- Section ID: `expand-mamba-ssm-code-2`
- Section: 실습 코드 2: SSM (State Space Model) 직접 구현

> 이 버전은 원본 실습 코드에 있던 들여쓰기 오류(실행 불가 상태였습니다)를 고치고, 개념 설명 · 손으로 계산해보는 예제 · 단계별 shape 추적 · 미니 실험을 추가한 학습용 버전입니다.

## 이 노트북에서 배우는 것

이 노트북은 "실행만 하고 끝"이 아니라, 수식이 실제 텐서 연산으로 바뀌는 과정을 한 줄씩 따라가기 위한 실습 노트입니다. 아래 순서로 진행됩니다.

1. **State Space Model(SSM)이 무엇인지** 비유(물탱크)를 통해 감을 잡습니다.
2. **아주 작은 숫자**로 SSM의 재귀식을 손으로 계산해봅니다. (torch 없이 순수 파이썬으로)
3. Mamba가 기존 S4와 다른 점, **"Selective(선택적)"**가 왜 중요한지 알아봅니다.
4. **이산화(discretization)** — 연속 시간 수식을 컴퓨터가 계산할 수 있는 형태로 바꾸는 과정을 알아봅니다.
5. 위 개념들을 실제 `nn.Module`인 `SimpleSSM`으로 옮기고, 조각조각 뜯어봅니다.
6. `MambaBlock`과 `TinyMambaLM`으로 쌓아 올려 실제 언어모델 형태를 만들어봅니다.
7. Attention과 비교해서 왜 Mamba가 긴 시퀀스에서 유리한지 실행 시간을 직접 재보며 확인합니다.

> 💡 **읽는 팁**: 코드 셀 위의 설명을 먼저 읽고, 코드 안의 주석을 따라가며 shape(텐서 모양)이 어떻게 바뀌는지 확인하세요. 처음 보는 용어는 코드보다 먼저 마크다운 설명에서 정의됩니다.

In [ ]:
# ============================================================
# 0. 준비물 (imports)
# ============================================================
# torch                    : 텐서 연산과 자동미분(autograd)을 담당
# torch.nn (nn)            : Linear, Conv1d, LayerNorm 같은 신경망 "레이어"를 만들 때 사용
# torch.nn.functional (F)  : silu, softplus처럼 "레이어가 아닌" 함수형 연산에 사용
# math, time               : 손 계산 예제와 실행 시간 측정에 사용
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

# 실행할 때마다 같은 난수가 나오도록 시드를 고정합니다.
# → 나중에 하이퍼파라미터를 바꿔가며 실험할 때 "값이 달라진 이유가 코드 때문인지, 그냥 랜덤이라 그런 건지" 헷갈리지 않기 위함입니다.
torch.manual_seed(42)

print("torch 버전:", torch.__version__)

## 1. State Space Model이란? — 물탱크로 이해하기

Transformer의 Attention은 "모든 토큰이 다른 모든 토큰을 한 번씩 서로 쳐다본다"는 방식으로 문맥을 파악합니다. 반면 State Space Model(SSM)은 **하나의 "상태(state)"를 계속 들고 다니면서 순서대로 업데이트**하는 방식으로 문맥을 파악합니다. RNN의 hidden state와 비슷하지만, 그 업데이트 규칙이 연속 시간(continuous-time) 미분방정식에서 출발한다는 점이 다릅니다.

### 비유: 물탱크

SSM에 나오는 4개의 행렬 A, B, C, D를 "물탱크"에 비유하면 아래처럼 생각할 수 있습니다.

| 기호 | 의미 | 물탱크 비유 |
|---|---|---|
| `h(t)` | 상태(state) | 물탱크의 **현재 수위** |
| `x(t)` | 입력(input) | 지금 이 순간 **새로 들어오는 물의 양** (= 지금 토큰의 정보) |
| `A` | 상태 전이 행렬 | 입력이 없어도 수위가 저절로 **줄어드는(새는) 속도** — "과거를 얼마나 오래 기억할지" 결정 |
| `B` | 입력 행렬 | 새로 들어오는 물이 수위에 **얼마나 반영될지** 조절하는 밸브 |
| `C` | 출력 행렬 | 수위계를 보고 **밖으로 알려줄 값**을 읽어내는 방법 |
| `D` | 직결 항 | 입력을 출력에 **직통으로 연결**해버리는 우회 파이프 (skip connection) |

연속 시간에서는 이렇게 씁니다.

```
h'(t) = A h(t) + B x(t)      # 수위의 "변화율" = 자연 감소분 + 새로 들어온 물
y(t)  = C h(t) + D x(t)      # 지금 출력 = 수위를 읽은 값 + 입력 직결분
```

그런데 컴퓨터는 "연속적으로 흐르는 시간"을 그대로 계산할 수 없고, `t = 1, 2, 3, ...`처럼 **이산적인 시간 단위(토큰 하나하나)**로만 계산할 수 있습니다. 그래서 위 식을 아래처럼 이산 시간(discrete-time) 버전으로 바꿔서 사용합니다.

```
h_k = Ā · h_{k-1} + B̄ · x_k     # k번째 시점의 새 수위
y_k = C · h_k + D · x_k         # k번째 시점의 출력
```

`Ā`(A bar), `B̄`(B bar)는 원래의 연속 시간 `A`, `B`를 "한 스텝만큼" 이산화한 버전입니다. 이 변환 과정을 **이산화(discretization)**라고 부르며, 3번 섹션에서 자세히 다룹니다.

지금은 아래 한 문장만 기억하면 됩니다.

> **SSM은 매 시점 t마다 "이전 상태 h_{t-1}"과 "지금 입력 x_t"를 받아서 "새 상태 h_t"를 만들고, 그 상태에서 "출력 y_t"를 읽어내는 것을 반복하는 모델입니다.**

이것은 RNN의 hidden state 업데이트와 거의 같은 모양입니다. Mamba가 특별한 이유는, 이 A, B, C, Δ(step size)를 **입력에 따라 달라지게(selective)** 만들었기 때문인데, 이는 2번 섹션에서 이어서 다룹니다.

In [ ]:
# ============================================================
# 손으로 따라가는 미니 SSM 예제 (torch 없이, 순수 파이썬)
# ============================================================
# 목표: "상태가 시간에 따라 어떻게 남았다가 사라지는지"를 실제 숫자로 확인합니다.
# 여기서는 상태(state)가 딱 1개짜리 숫자(d_state=1)라고 가정합니다. (물탱크 1개)

A = -0.5   # 상태 전이 계수. 음수 = "가만히 두면 수위가 자연스럽게 줄어든다"는 뜻
B = 1.0    # 입력이 상태에 반영되는 비율
C = 1.0    # 상태를 출력으로 읽어내는 비율
D = 0.0    # 입력이 출력에 직접 더해지는 비율 (일단 0으로 꺼둡니다)
dt = 1.0   # 이산화 스텝 크기 Δ (토큰 하나가 지날 때마다의 "시간 간격"이라고 생각하면 됩니다)

# 이산화 (자세한 유도는 3번 섹션에서)
A_bar = math.exp(dt * A)   # Ā = exp(Δ·A)
B_bar = dt * B             # B̄ ≈ Δ·B  (오일러 근사. 실제 Mamba는 조금 더 정교한 식을 씁니다)

print(f"이산화 결과 → Ā = {A_bar:.4f}, B̄ = {B_bar:.4f}\n")

# 입력 시퀀스: 1번째와 4번째 시점에만 "impulse"(자극)를 주고 나머지는 0으로 둡니다.
# → 이렇게 하면 "입력이 없을 때 상태가 저절로 어떻게 사라지는지"가 잘 보입니다.
inputs = [1.0, 0.0, 0.0, 2.0, 0.0]

h = 0.0  # 초기 상태 (물탱크가 텅 빈 상태에서 시작)
print(f"{'t':>2} | {'x_t':>6} | {'h_t (상태)':>10} | {'y_t (출력)':>10}")
print("-" * 40)
for t, x_t in enumerate(inputs, start=1):
    h = A_bar * h + B_bar * x_t      # h_k = Ā·h_{k-1} + B̄·x_k
    y_t = C * h + D * x_t            # y_k = C·h_k + D·x_k
    print(f"{t:>2} | {x_t:>6.1f} | {h:>10.4f} | {y_t:>10.4f}")

print("\n관찰: 1번째 시점에 준 자극(1.0)이 곧바로 사라지지 않고,")
print("      Ā(≈0.61)의 비율로 서서히 줄어들며 여러 시점에 걸쳐 '메아리'처럼 남습니다.")
print("      이것이 SSM이 '과거 정보를 기억'하는 방식입니다. A를 더 0에 가깝게 두면 더 오래 남고,")
print("      더 음수로(예: -3.0) 두면 훨씬 빨리 사라지겠죠 — 아래 '직접 실험해보기'에서 시도해보세요.")

### 지금까지는 "물탱크 1개"였습니다 — 실제로는 이렇게 확장됩니다

방금 예제는 상태가 숫자 하나(`h`)뿐인 가장 단순한 경우였습니다. 실제 Mamba에서는 이걸 세 가지 축으로 확장합니다.

1. **`d_state`** (예: 16개) : 물탱크 1세트 안에 여러 개의 "메모리 슬롯"을 둡니다. 어떤 슬롯은 빨리 잊고(A가 크게 음수), 어떤 슬롯은 오래 기억하도록(A가 0에 가까움) 서로 다른 속도로 설계합니다.
2. **`d_inner`** (채널 수, 예: 512) : 위 물탱크 "세트"를 채널(특징 차원)마다 독립적으로 둡니다. 즉 채널마다 자기만의 `d_state`개짜리 메모리를 따로 가집니다.
3. **`batch`** : 여러 개의 시퀀스(문장)를 동시에 처리하기 위한 축입니다.

그래서 실제 코드에서 상태 `h`는 숫자 하나가 아니라 **`(batch, d_inner, d_state)`** shape의 텐서가 됩니다. 아래부터는 이 shape이 코드 곳곳에서 계속 등장하니 눈여겨봐주세요.

## 2. "Selective(선택적)" 메커니즘 — S4와 Mamba의 결정적 차이

방금 예제에서 A, B, C, D, Δ는 전부 **고정된 숫자**였습니다. 이것이 바로 Mamba 이전 모델인 S4(Structured State Space)가 동작하는 방식입니다: 한 번 학습되고 나면, 모든 입력에 대해 항상 같은 A, B, C, Δ를 사용합니다.

문제는, 언어에는 "이 단어는 중요하니까 오래 기억해야 하고, 이 단어(예: 관사 'the')는 중요하지 않으니 금방 잊어도 된다" 같은 **내용에 따른 선택**이 필요하다는 점입니다. A, B, Δ가 고정이면 모든 토큰을 항상 같은 방식으로 취급할 수밖에 없습니다.

Mamba의 핵심 아이디어(그래서 정식 명칭이 Selective State Space Model, 줄여서 S6)는 다음과 같습니다.

> **B, C, Δ를 고정된 파라미터가 아니라, "지금 들어온 입력 x_t가 무엇이냐"에 따라 매번 다시 계산되는 값으로 만들자.**

즉, 아래처럼 입력 `x_t`를 작은 Linear 레이어에 통과시켜서 그 시점의 B, C, Δ를 즉석에서 만들어냅니다.

```
B_t, C_t, Δ_t  =  Linear(x_t)   # 매 시점마다 입력에 따라 새로 계산!
```

이렇게 하면 모델이 "이 토큰은 중요한 정보니까 Δ를 크게 잡아서 많이 반영하고 오래 남기자" 또는 "이 토큰은 중요하지 않으니 Δ를 작게 잡아서 거의 무시하자" 같은 판단을 **내용을 보고** 할 수 있게 됩니다. Attention이 "각 토큰끼리 얼마나 관련있는지"를 모든 쌍에 대해 직접 비교(N×N)해서 선택하는 것과 달리, Mamba는 **한 번의 순차적인 스캔만으로** 이런 내용 기반 선택을 해낸다는 점이 핵심입니다.

(참고: A 자체는 코드에서 고정 파라미터로 두지만, Δ가 입력마다 달라지기 때문에 실질적으로 "매 시점 다른 속도로 상태가 갱신"되는 효과를 냅니다. 이산화된 Ā = exp(Δ·A)이므로, Δ가 달라지면 Ā도 함께 달라지기 때문입니다.)

아래 코드에서 `x_proj`가 B, C를 만들고 `dt_proj`가 Δ를 만드는 부분이 바로 이 "selective" 메커니즘입니다.

## 3. 이산화(Discretization) 조금 더 자세히 — Zero-Order Hold(ZOH)

앞서 이산화를 이렇게 정의했습니다.

```
Ā = exp(Δ · A)
B̄ ≈ Δ · B
```

`Ā`는 정확한 식입니다. 그런데 `B̄`의 "정확한" 식은 사실 조금 더 복잡합니다 (Zero-Order Hold 방식):

```
B̄_정확 = (Δ·A)⁻¹ · (exp(Δ·A) − I) · Δ·B
```

이 노트북(그리고 많은 "간단 구현" 튜토리얼들)은 계산을 단순화하기 위해 **오일러 근사(Euler approximation)**인 `B̄ ≈ Δ·B`를 사용합니다. Δ가 충분히 작을 때는 두 식의 차이가 크지 않고, 무엇보다 **"shape이 어떻게 바뀌고 정보가 어떻게 흘러가는지"를 이해하는 것이 이 실습의 목적**이므로 오일러 근사로도 충분합니다.

> 🔎 **심화 참고**: 논문 저자들이 공개한 공식 Mamba 구현체(`mamba-ssm` 패키지)는 위의 정확한 ZOH 공식을 CUDA 커널로 최적화해서 사용합니다. 지금은 "그런 게 있다" 정도만 알아두고, 이 노트북에서는 이해하기 쉬운 오일러 근사 버전으로 계속 진행합니다.

정리하면, **이산화란 연속 시간 미분방정식(`h'(t) = ...`)을, 한 스텝(Δ)마다 상태를 얼마나 업데이트할지 정하는 곱셈·덧셈 규칙(`h_k = Ā·h_{k-1} + B̄·x_k`)으로 바꾸는 과정**입니다. 아래 코드에서는 이 Δ가 고정값이 아니라 `dt_proj`를 통해 입력마다 다르게 계산된다는 점이 (2번 섹션에서 설명한) selective 메커니즘과 만나는 지점입니다.

## 4. 이제 진짜 코드로: `SimpleSSM` 클래스 만들기

지금까지 배운 개념을 실제 `nn.Module`로 옮겨보겠습니다. 먼저 `__init__`에 어떤 레이어들이 필요한지, 그리고 각 레이어가 "물탱크 비유"의 어떤 부분에 해당하는지 표로 정리했습니다.

| 레이어 | 역할 | 비유 / 설명 |
|---|---|---|
| `in_proj` | 입력을 두 갈래로 나눠 투영 | 하나는 SSM으로 갈 "내용" 갈래(`x_branch`), 하나는 게이트로 쓸 "밸브" 갈래(`z`) |
| `conv1d` | 짧은 구간의 지역 패턴 포착 | 바로 이전 몇 단어(`d_conv`개)를 슬쩍 미리 섞어보는 역할. n-gram 필터와 비슷 |
| `x_proj` | 입력에서 B, C, Δ(의 원재료)를 뽑아냄 | 2번 섹션의 "selective" 파라미터 생성기 |
| `dt_proj` | Δ(스텝 크기)를 채널별로 최종 계산 | 채널마다 "이 순간을 얼마나 크게 반영할지" 결정 |
| `A_log` | 상태 전이 행렬 A (log로 저장) | 물탱크가 "저절로 새는 속도" — 채널·상태 슬롯마다 다르게 초기화 |
| `D` | 입력→출력 직결 계수 | 우회 파이프 (skip connection) |
| `out_proj` | 다시 원래 차원(`d_model`)으로 투영 | SSM 처리가 끝난 결과를 다음 블록이 받을 수 있는 크기로 되돌림 |

> ℹ️ **왜 `A`를 바로 저장하지 않고 `A_log`(로그 값)로 저장할까요?** SSM이 안정적으로 동작하려면 `A`가 항상 음수여야 합니다 (그래야 물탱크가 무한히 차오르지 않고 "새어나가는" 방향으로만 작동합니다). `A = -exp(A_log)`로 계산하면, `A_log`가 어떤 실수값을 갖든 `exp(A_log)`는 항상 양수이므로 `A`는 항상 음수가 되는 것이 자동으로 보장됩니다. 이렇게 "특정 부호를 강제하기 위해 log/exp를 거치는" 트릭은 여러 딥러닝 코드에서 자주 등장합니다.

`conv1d` 부분은 살짝 헷갈릴 수 있는 트릭(causal padding)이 들어있어서, 전체 클래스를 보기 전에 미니 실험으로 먼저 짚고 넘어가겠습니다.

In [ ]:
# ============================================================
# 잠깐! conv1d가 왜 이렇게 생겼는지 미니 실험으로 확인하고 넘어가기
# ============================================================
# SimpleSSM 안의 conv1d는 "미래를 훔쳐보지 않는" causal(인과적) 컨볼루션이어야 합니다.
# 그런데 nn.Conv1d는 기본적으로 패딩을 "양쪽에 동일하게" 붙입니다.
# 그래서 "패딩을 조금 더 많이 주고, 뒤쪽 결과를 잘라내는" 트릭으로 causal하게 만듭니다.
# 아래에서 아주 작은 예시로 직접 확인해봅시다.

demo_conv = nn.Conv1d(
    in_channels=1, out_channels=1,
    kernel_size=4,          # d_conv=4라고 가정: "직전 4개 시점"을 보는 필터
    padding=4 - 1,           # = 3. 양쪽에 3칸씩 0으로 채웁니다
    bias=False,
)
# 커널 가중치를 눈으로 확인하기 쉽게 전부 1로 고정 (그냥 "최근 4개 값의 합"을 구하는 필터가 되도록)
with torch.no_grad():
    demo_conv.weight.fill_(1.0)

# 입력: 길이 6짜리 시퀀스, 값은 1,2,3,4,5,6 (shape: batch=1, channel=1, length=6)
demo_x = torch.arange(1, 7).float().view(1, 1, 6)
demo_out_full = demo_conv(demo_x)
print("입력 길이                 :", demo_x.shape[-1])
print("패딩 붙인 뒤 conv 결과 길이 :", demo_out_full.shape[-1], "← 원래보다 길어짐 (양쪽 패딩 때문)")

demo_out_causal = demo_out_full[:, :, :demo_x.shape[-1]]  # 앞에서부터 원래 길이만큼만 자름
print("앞에서부터 잘라낸 causal 결과:", demo_out_causal.squeeze().tolist())

print("\n확인: 첫 번째 출력값은 1인데, 이는 [패딩 0, 0, 0] + [입력의 첫 값 1]의 합입니다.")
print("      즉 1번째 시점의 출력은 '미래(2,3,4,5,6)를 전혀 보지 않고' 자기 자신까지만 사용했다는 뜻입니다.")
print("      만약 뒤쪽 6개를 잘랐다면 미래 값이 섞여 들어와 causal하지 않게 됩니다.")
print("      SimpleSSM.forward()의 `x_conv = self.conv1d(...)[:, :, :seq_len]` 부분이 바로 이 '앞에서부터 자르기'입니다.")

In [ ]:
class SimpleSSM(nn.Module):
    """Selective State Space Model (Mamba의 핵심 블록)을 간소화한 구현.

    연속 시간 :  h'(t) = A h(t) + B x(t),    y(t) = C h(t) + D x(t)
    이산 시간 :  h_k   = Ā h_{k-1} + B̄ x_k,   y_k  = C h_k + D x_k

    위 1~4번 섹션에서 다룬 내용을 그대로 코드로 옮긴 것입니다.
    각 줄의 shape 주석 (batch, seq, ...)을 계속 눈으로 따라가면서 읽어주세요.
    """

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        """
        Args:
            d_model : 이 블록에 들어오고 나가는 벡터의 차원 (Transformer의 hidden size와 같은 역할)
            d_state : 채널 하나당 갖는 '메모리 슬롯' 개수 (물탱크 세트 안의 물탱크 개수)
            d_conv  : 1D 컨볼루션이 한 번에 들여다보는 최근 시점 개수 (지역 문맥 크기)
            expand  : 내부 계산을 몇 배 넓은 차원(d_inner)에서 할지 정하는 배율
        """
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = int(d_model * expand)  # 예: d_model=256, expand=2 → d_inner=512

        # ── ① 입력 프로젝션: d_model → d_inner를 "두 벌" 만든다 ──
        # 결과를 반으로 나눠서 x_branch(SSM으로 갈 내용)와 z(게이트 밸브)로 사용합니다.
        # d_model → d_inner를 두 번 따로 하는 것과 수학적으로는 같지만, 하나의 큰 행렬
        # 곱셈으로 합쳐서 계산하면 GPU에서 더 효율적이기 때문에 이렇게 구현합니다.
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # ── ② 1D 컨볼루션: 최근 d_conv개 시점의 지역 문맥을 살짝 섞어준다 ──
        # groups=self.d_inner ⇒ "depthwise convolution": 채널끼리 서로 섞지 않고,
        # 채널 각각을 자기 자신의 과거 값들하고만 컨볼루션합니다 (파라미터 수를 크게 절약).
        # padding=d_conv-1로 넉넉히 패딩한 뒤, forward에서 앞부분만 잘라 causal하게 만듭니다.
        # (바로 위 셀의 미니 실험에서 이 트릭을 직접 확인했습니다.)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner, bias=True
        )

        # ── ③ Selective 파라미터 생성기: 입력 x로부터 B, C, (raw)Δ를 만든다 ──
        # 출력 차원이 d_state*2 + 1인 이유: [B에 쓸 d_state개] + [C에 쓸 d_state개] + [Δ의 원재료 1개]
        # 이 레이어가 2번 섹션에서 설명한 "Selective(선택적)" 메커니즘의 핵심입니다.
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)

        # ── ④ dt(Δ, step size) 프로젝션 ──
        # x_proj가 만든 "Δ 원재료"(채널 공통 스칼라 1개)를 받아서, 채널(d_inner)마다
        # 서로 다른 Δ값을 갖도록 확장합니다. 채널마다 "이 순간을 얼마나 크게 반영할지"가 달라집니다.
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        # ── ⑤ A 파라미터 (상태 전이 행렬, 대각 성분만 사용) ──
        # shape: (d_inner, d_state) → 채널마다, 상태 슬롯마다 서로 다른 "감쇠 속도"를 가짐.
        # torch.arange(1, d_state+1) = [1, 2, ..., d_state]로 초기화하는 이유:
        #   슬롯 1번은 A≈-1 (천천히 잊음), 슬롯 d_state번은 A≈-d_state (빨리 잊음)처럼
        #   슬롯마다 "서로 다른 기억 지속 시간"을 갖도록 다양성을 주기 위함입니다.
        # 실제 값은 forward에서 -exp(A_log)로 계산합니다. (아래 A_log 관련 설명 참고)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(self.d_inner, 1)))

        # ── ⑥ D 파라미터 (skip connection) ──
        # 입력이 SSM 상태를 거치지 않고 출력에 곧바로 더해지는 "직결 파이프". 1로 초기화합니다.
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # ── ⑦ 출력 프로젝션: 다시 d_model 차원으로 되돌린다 ──
        # 다음 MambaBlock이 같은 크기의 입력을 받을 수 있도록 맞춰주는 역할입니다.
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x, verbose=False):
        """x: (batch, seq_len, d_model) → (batch, seq_len, d_model)"""
        batch, seq_len, _ = x.shape
        if verbose:
            print(f"  [forward] 입력 x             : {tuple(x.shape)}  (batch, seq_len, d_model)")

        # 1) 입력 프로젝션 + 두 갈래로 쪼개기
        xz = self.in_proj(x)                 # (batch, seq, d_inner*2)
        x_branch, z = xz.chunk(2, dim=-1)     # 각각 (batch, seq, d_inner)
        if verbose:
            print(f"  [forward] in_proj 출력 xz     : {tuple(xz.shape)} → x_branch {tuple(x_branch.shape)}, z(게이트) {tuple(z.shape)}")

        # 2) 1D 컨볼루션으로 지역 문맥 포착 (causal하게 앞부분만 사용)
        #    Conv1d는 (batch, channel, length) 순서를 기대하므로 transpose로 축을 맞춰줍니다.
        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x_act = F.silu(x_conv)  # SiLU(x) = x * sigmoid(x). ReLU보다 부드럽게 이어지는 활성화 함수
        if verbose:
            print(f"  [forward] conv+SiLU 출력      : {tuple(x_act.shape)}  (지역 문맥이 섞인 상태)")

        # 3) SSM 순차 스캔 (이 블록의 핵심 — 바로 아래 _ssm_scan에서 자세히 설명)
        y = self._ssm_scan(x_act, verbose=verbose)

        # 4) 게이트 적용: z가 "이 정보를 얼마나 통과시킬지" 정하는 밸브 역할을 합니다.
        #    F.silu(z)가 0에 가까우면 y를 거의 막고, 1에 가까우면 y를 거의 그대로 통과시킵니다.
        y = y * F.silu(z)
        if verbose:
            print(f"  [forward] 게이트(z) 적용 출력  : {tuple(y.shape)}")

        # 5) 출력 프로젝션으로 d_model 차원으로 복귀
        out = self.out_proj(y)
        if verbose:
            print(f"  [forward] 최종 출력           : {tuple(out.shape)}  (batch, seq_len, d_model)")
        return out

    def _ssm_scan(self, x, verbose=False):
        """SSM 순차 스캔 (recurrent mode). x: (batch, seq_len, d_inner)"""
        batch, seq_len, d_inner = x.shape

        # ── selective 파라미터 계산 (2번 섹션 참고: 입력에 따라 매번 달라집니다!) ──
        x_dbl = self.x_proj(x)                            # (batch, seq, d_state*2 + 1)
        B = x_dbl[:, :, :self.d_state]                    # (batch, seq, d_state)
        C = x_dbl[:, :, self.d_state:self.d_state * 2]    # (batch, seq, d_state)
        # softplus(z) = log(1+exp(z)) : 항상 양수를 반환하는 부드러운 함수.
        # Δ(step size)는 항상 양수여야 의미가 있으므로(시간이 거꾸로 흐를 순 없으니) softplus를 사용합니다.
        dt = F.softplus(self.dt_proj(x_dbl[:, :, -1:]))   # (batch, seq, d_inner)

        if verbose:
            print(f"    [_ssm_scan] x_proj 출력 x_dbl : {tuple(x_dbl.shape)}")
            print(f"    [_ssm_scan] B {tuple(B.shape)}  C {tuple(C.shape)}  dt {tuple(dt.shape)}")

        # A를 계산 (항상 음수가 되도록 -exp(...) 사용 — __init__의 A_log 설명 참고)
        A = -torch.exp(self.A_log)  # (d_inner, d_state)

        # 상태 h를 0(빈 물탱크)에서 시작
        h = torch.zeros(batch, d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []

        for t in range(seq_len):
            # ── 이 시점(t)의 Δ로 Ā, B̄를 계산 (오일러 근사, 3번 섹션 참고) ──
            dt_t = dt[:, t, :].unsqueeze(-1)     # (batch, d_inner, 1)

            # A는 (d_inner, d_state)로 batch 차원이 없지만, dt_t는 (batch, d_inner, 1)로 batch 차원이 있습니다.
            # PyTorch의 broadcasting 규칙(뒤쪽 차원부터 비교해서 크기가 1이거나 같으면 자동으로 맞춰줌) 덕분에
            # A가 모든 batch에 대해 자동으로 "복사"된 것처럼 곱해집니다. → 결과는 batch 차원이 생긴 (batch, d_inner, d_state)
            dA = torch.exp(A * dt_t)             # Ā_t = exp(Δ_t · A)   → (batch, d_inner, d_state)
            dB = B[:, t, :].unsqueeze(1) * dt_t  # B̄_t ≈ Δ_t · B_t      → (batch, d_inner, d_state) (마찬가지로 broadcasting)

            # ── 상태 갱신: h_t = Ā_t · h_{t-1} + B̄_t · x_t ──
            h = h * dA + dB * x[:, t, :].unsqueeze(-1)

            # ── 출력 계산: y_t = C_t · h_t + D · x_t ──
            y_t = (h * C[:, t, :].unsqueeze(1)).sum(-1) + self.D * x[:, t, :]
            ys.append(y_t)

            if verbose and t < 2:  # 앞의 2 스텝만 예시로 출력 (전부 찍으면 너무 길어지므로)
                print(f"    [_ssm_scan] t={t} → dA {tuple(dA.shape)}, h {tuple(h.shape)}, y_t {tuple(y_t.shape)}")

        return torch.stack(ys, dim=1)  # (batch, seq, d_inner)

## 5. 실제로 shape이 어떻게 흘러가는지 눈으로 확인하기

`verbose=True`로 호출하면 `forward` 안에서 각 단계 텐서의 shape을 출력하도록 만들어 두었습니다.
아주 작은 크기(`d_model=8, d_state=4, seq_len=3, batch=2`)로 실행해서, 지금까지 설명한 shape들이 실제로 어떻게 나오는지 확인해봅시다.

In [ ]:
# 아주 작은 크기로 SimpleSSM 하나를 직접 실행해서 shape을 눈으로 확인합니다.
tiny_ssm = SimpleSSM(d_model=8, d_state=4, d_conv=4, expand=2)
tiny_x = torch.randn(2, 3, 8)  # batch=2, seq_len=3, d_model=8

print("=== SimpleSSM.forward(verbose=True) 실행 ===")
tiny_out = tiny_ssm(tiny_x, verbose=True)

print(f"\n입력 shape : {tuple(tiny_x.shape)}")
print(f"출력 shape : {tuple(tiny_out.shape)}")
print("→ 입력과 출력의 shape이 동일합니다 (batch, seq_len, d_model).")
print("  SSM 블록은 shape을 바꾸지 않고 '내용'만 문맥 정보로 갱신해서 돌려줍니다.")
print("  (Transformer에서 attention block이 하는 역할과 같습니다.)")

## 6. Residual + LayerNorm으로 감싸기: `MambaBlock`

신경망을 깊게 쌓을수록 학습이 불안정해지는 문제가 있습니다. Transformer가 이를 해결하기 위해 "LayerNorm + Residual(잔차 연결)"을 사용하는 것처럼, Mamba도 동일한 방식을 사용합니다.

```
out = x + SSM(LayerNorm(x))
```

- **LayerNorm(x)** : SSM에 들어가기 전에 값의 크기(scale)를 일정하게 정규화해서 학습을 안정시킵니다. SSM *이전에* 정규화하므로 이런 구조를 "Pre-Norm"이라고 부릅니다.
- **x + (...)** : SSM이 계산한 결과를 원래 입력에 "더해서" 반환합니다. 이렇게 하면 SSM이 아직 유용한 것을 학습하지 못한 초기 단계라도(출력이 0에 가까워도) 최소한 원래 입력 정보(x)는 그대로 다음 층에 전달됩니다. 즉, 정보가 사라지지 않도록 보장하는 "지름길"입니다.

이 두 가지를 합친 것이 `MambaBlock`이며, 이 블록을 여러 개 쌓으면 하나의 모델이 됩니다.

In [ ]:
class MambaBlock(nn.Module):
    """Mamba 블록: SSM + Residual + Pre-Norm

    구조: out = x + SSM(LayerNorm(x))
    Transformer의 '한 층(layer)'에 해당하는 단위이며, 이 블록을 여러 개 쌓아 깊은 모델을 만듭니다.
    """

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)              # SSM에 들어가기 전 정규화 (Pre-Norm)
        self.ssm = SimpleSSM(d_model, d_state, d_conv, expand)

    def forward(self, x, verbose=False):
        # 잔차 연결(residual connection): 입력 x를 SSM 출력에 그대로 더해줍니다.
        return x + self.ssm(self.norm(x), verbose=verbose)

In [ ]:
# ============================================================
# 단일 블록 테스트
# ============================================================
print("=== SSM (Mamba 스타일) 단일 블록 테스트 ===\n")

block = MambaBlock(d_model=256, d_state=16, d_conv=4, expand=2)
x = torch.randn(2, 64, 256)  # batch=2, seq_len=64, d_model=256 (문장 2개, 각 64토큰이라고 생각하면 됩니다)
out = block(x)

params = sum(p.numel() for p in block.parameters())
print(f"입력 shape  : {tuple(x.shape)}  (batch, seq_len, d_model)")
print(f"출력 shape  : {tuple(out.shape)}")
print(f"파라미터 수 : {params:,}개")

## 7. 여러 블록을 쌓아 언어모델 만들기: `TinyMambaLM`

`MambaBlock` 하나는 "문맥을 반영해서 벡터를 갱신"하는 역할만 합니다. 이를 실제 언어모델로 만들려면 GPT류 Transformer 모델과 동일한 뼈대가 필요합니다.

```
토큰 ID  →  [Embedding]  →  [MambaBlock] × N층  →  [LayerNorm]  →  [Linear head]  →  다음 토큰 점수
```

- **Embedding** : 정수 토큰 ID(예: "고양이" = 1523번)를 `d_model` 차원의 벡터로 바꿉니다.
- **MambaBlock × N층** : 앞서 만든 블록을 N번 통과시키면서 점점 더 풍부한 문맥 정보를 벡터에 담습니다.
- **최종 LayerNorm** : 마지막에 한 번 더 정규화합니다 (Transformer 계열 모델들의 관례적인 마무리 단계).
- **head (Linear)** : `d_model` 차원 벡터를 다시 `vocab_size` 차원으로 투영해서, "다음 토큰이 무엇일지"에 대한 단어별 점수(logit)를 계산합니다.

Attention 대신 Mamba 블록을 쓴다는 점만 다를 뿐, "임베딩 → 블록 반복 → 다음 토큰 예측"이라는 전체 뼈대는 GPT와 동일합니다.

In [ ]:
class TinyMambaLM(nn.Module):
    """MambaBlock을 여러 층 쌓은 아주 작은 언어모델.

    입력: 토큰 ID 시퀀스 (batch, seq_len) — 정수
    출력: 다음 토큰 예측을 위한 logit (batch, seq_len, vocab_size)
    """

    def __init__(self, vocab_size=32000, d_model=256, n_layers=4, d_state=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)          # 토큰 ID → 벡터
        self.blocks = nn.ModuleList([
            MambaBlock(d_model, d_state) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)                        # 마지막 정규화
        self.head = nn.Linear(d_model, vocab_size, bias=False)   # 벡터 → 단어별 점수(logit)

    def forward(self, idx, verbose=False):
        x = self.embed(idx)  # (batch, seq_len) → (batch, seq_len, d_model)
        for i, block in enumerate(self.blocks):
            # 첫 번째 블록만 verbose 출력 (모든 층에서 다 찍으면 너무 길어지므로)
            x = block(x, verbose=(verbose and i == 0))
        return self.head(self.norm(x))  # (batch, seq_len, vocab_size)

In [ ]:
# ============================================================
# 작은 Mamba 언어모델 테스트
# ============================================================
model = TinyMambaLM(vocab_size=32000, d_model=256, n_layers=4)
params = sum(p.numel() for p in model.parameters())

x = torch.randint(0, 32000, (2, 128))  # batch=2, seq_len=128짜리 랜덤 토큰 ID (실제로는 진짜 문장이 들어갑니다)
out = model(x)

print(f"TinyMambaLM 입력 shape : {tuple(x.shape)}  (batch, seq_len) — 토큰 ID")
print(f"TinyMambaLM 출력 shape : {tuple(out.shape)}  (batch, seq_len, vocab_size) — 위치별 '다음 토큰' 점수")
print(f"전체 파라미터 수        : {params:,}개")

## 8. Attention과 비교해보기: 왜 "O(N)"이 중요한가

Transformer의 Self-Attention은 시퀀스 길이가 N일 때 **모든 토큰 쌍(N×N개)**의 관련도를 계산해야 하므로, 연산량이 시퀀스 길이의 제곱(**O(N²)**)에 비례해서 늘어납니다. 문장이 2배 길어지면 계산량은 4배가 됩니다.

반면 SSM은 토큰을 **한 번씩만** 순서대로 지나가며 상태를 갱신하므로 연산량이 시퀀스 길이에 비례(**O(N)**)합니다. 문장이 2배 길어지면 계산량도 딱 2배만 늘어납니다.

말로만 하면 추상적이니, 시퀀스 길이를 늘려가며 `MambaBlock`을 실행하는 데 걸리는 시간을 실제로 측정해서 "정말 선형적으로 늘어나는지" 확인해봅시다.

> 참고: 아래 측정은 파이썬 `for`문으로 구현된 `_ssm_scan` 기준이라, 논문에서 말하는 하드웨어 최적화된 병렬 스캔보다는 느립니다. 그래도 "시퀀스 길이에 비례해서 늘어난다"는 경향성은 동일하게 관찰할 수 있습니다.

In [ ]:
# ============================================================
# 시퀀스 길이에 따른 실행 시간 측정 (O(N) 확인)
# ============================================================
print("=== 시퀀스 길이에 따른 실행 시간 측정 ===\n")
timing_block = MambaBlock(d_model=128, d_state=16, d_conv=4, expand=2)
timing_block.eval()

seq_lengths = [64, 128, 256, 512]
times = []

with torch.no_grad():
    for L in seq_lengths:
        x = torch.randn(1, L, 128)

        # 워밍업 1회 (torch 내부 캐시 등의 영향을 배제하기 위함)
        _ = timing_block(x)

        # 잡음(noise)을 줄이기 위해 3번 측정해서 가장 빠른 시간을 사용합니다.
        trial_times = []
        for _ in range(3):
            start = time.time()
            _ = timing_block(x)
            trial_times.append(time.time() - start)
        best = min(trial_times)

        times.append(best)
        print(f"seq_len={L:>4} → {best * 1000:.2f} ms")

print("\n길이가 2배가 될 때마다 시간이 몇 배가 되는지 확인:")
for i in range(1, len(seq_lengths)):
    ratio_len = seq_lengths[i] / seq_lengths[i - 1]
    ratio_time = times[i] / times[i - 1] if times[i - 1] > 0 else float("nan")
    print(f"  길이 {seq_lengths[i-1]:>3} → {seq_lengths[i]:>3}  ({ratio_len:.0f}배)   시간은 {ratio_time:.2f}배")

print("\n→ Attention이었다면 길이가 2배일 때 계산량은 이론상 4배(2²)에 가깝게 늘어납니다.")
print("  SSM은 길이에 '비례'해서 늘어나야 하므로, 위 배수가 4배보다는 2배 쪽에 훨씬 가까운 것을 확인할 수 있습니다.")
print("  (파이썬 for-loop와 시스템 상황에 따른 오차가 있어 정확히 2.00배가 아닐 수 있습니다 — 경향성 위주로 봐주세요.)")

## 정리 및 다음 실습을 위한 제안

이 노트북의 핵심 흐름을 한 문장으로 요약하면 다음과 같습니다.

> **입력을 두 갈래로 나누고(내용/게이트) → 내용 갈래는 conv로 지역 문맥을 살짝 섞은 뒤 → 입력에 따라 달라지는 B, C, Δ로 상태를 순차적으로 갱신(SSM scan)하고 → 게이트로 얼마나 통과시킬지 정한 뒤 → 다시 원래 차원으로 되돌린다.**

### 🧪 직접 실험해보기

아래 값들을 바꿔가며 위 셀들을 다시 실행해보고, 무엇이 달라지는지 관찰해보세요.

- `d_state`를 16 → 4 또는 64로 바꾸면 파라미터 수와 "기억할 수 있는 정보량"이 어떻게 달라질까요?
- `d_conv`를 4 → 1로 바꾸면(지역 컨볼루션을 사실상 없앤다면) 모델이 여전히 잘 동작할까요? 어떤 정보를 놓치게 될까요?
- `expand`를 2 → 4로 늘리면 파라미터 수가 정확히 몇 배가 되나요? (힌트: `in_proj`, `x_proj`, `out_proj` 모두 `d_inner`에 비례합니다.)
- 1번 섹션의 손 계산 예제에서 `A` 값을 `-0.1`이나 `-3.0`으로 바꿔보면, 자극(impulse)이 얼마나 오래/빨리 사라지나요?

### 📌 이 구현이 "간소화" 버전인 이유 (실제 Mamba와의 차이)

- `B̄`를 오일러 근사(`Δ·B`)로 계산합니다. 실제로는 더 정확한 ZOH 공식을 사용합니다. (3번 섹션 참고)
- `_ssm_scan`이 파이썬 `for`문으로 순차 처리합니다. 실제 Mamba는 이를 병렬화된 CUDA 스캔 커널로 구현해 훨씬 빠릅니다.
- `A`의 초기화가 `[1, 2, ..., d_state]`로 단순화되어 있습니다. 실제로는 HiPPO 이론에 기반한 좀 더 정교한 초기화를 사용합니다.

이런 차이들은 "왜 그렇게 동작하는지"에 대한 직관을 해치지 않으면서 코드를 단순하게 유지하기 위한 것입니다. 이 노트북으로 전체 흐름을 확실히 이해했다면, 다음 단계로 공식 Mamba 논문이나 `mamba-ssm` 공식 구현체를 읽어보는 것을 추천합니다.